# Configuración de paths e imports

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

from dataclasses import asdict
import cv2 as cv
import matplotlib.pyplot as plt

from Code.image.ImgPreproc import ImgPreproc, ImgPreprocCfg

intento_numero = 1
index = 10
labels = ["Tornillo", "Tuerca", "Arandela", "Clavo"]

guardar = True
batch_mode = False
mostrar = True


# Configuración del preprocesador

In [ ]:
cfg = ImgPreprocCfg(
    target_size=(256, 256),
    clahe_clip=2.0,
    clahe_grid=(8, 8),
    bilateral_d=9,
    bilateral_sigma_color=75,
    bilateral_sigma_space=75,
    canny_ratio=0.33,
    morph_kernel_size=(3, 3),
)
preproc = ImgPreproc(cfg)


# Ejecución del pipeline y visualización

In [ ]:
if not batch_mode:
    categorias = ["Tuerca"]
else:
    categorias = labels

output_root = PROJECT_ROOT / "DataBase" / "tmp" / f"ImgPreproc{intento_numero}"
output_root.mkdir(parents=True, exist_ok=True)

for categoria in categorias:
    categoria_dir = output_root / categoria
    categoria_dir.mkdir(parents=True, exist_ok=True)

    config_path = categoria_dir / "config.txt"
    with config_path.open("w", encoding="utf-8") as fh:
        for key, value in asdict(cfg).items():
            fh.write(f"{key} = {value}\n")

    for i in range(index):
        img_path = PROJECT_ROOT / "DataBase" / "data" / "images1" / f"{categoria}" /f"{categoria}{i+1:03d}.jpg"
        img_bgr = cv.imread(str(img_path))
        if img_bgr is None:
            raise FileNotFoundError(f"No pude leer {img_path}")

        try:
            cropped, mask, meta = preproc.process(img_path)
        except ValueError as exc:
            print(f"[WARN] {img_path.name}: {exc}")
            continue

        if guardar:
            cv.imwrite(str(categoria_dir / img_path.name), cropped)
            cv.imwrite(str(categoria_dir / f"{img_path.stem}_mask.png"), mask)

        if mostrar:
            fig, axs = plt.subplots(1, 3, figsize=(12, 4))
            axs[0].imshow(cv.cvtColor(img_bgr, cv.COLOR_BGR2RGB))
            axs[0].set_title("Original")
            axs[0].axis("off")

            axs[1].imshow(mask, cmap="gray")
            axs[1].set_title("Máscara")
            axs[1].axis("off")

            if cropped.ndim == 2:
                axs[2].imshow(cropped, cmap="gray")
            else:
                axs[2].imshow(cv.cvtColor(cropped, cv.COLOR_BGR2RGB))
            axs[2].set_title("Recorte normalizado")
            axs[2].axis("off")

            plt.tight_layout()
            plt.show()
            plt.close(fig)
